In [1]:

# ============================================================
# PROJECT 2: CUSTOMER CHURN & RETENTION INTELLIGENCE
# Interactive AI Agent | Purple + Teal Theme
# ============================================================

!pip -q install -U gradio pandas numpy scikit-learn plotly

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score

# -------------------- 1. GENERATE SAMPLE DATA --------------------

np.random.seed(42)
N = 1800

df = pd.DataFrame({
    "Customer_ID": [f"CUST-{i:04d}" for i in range(1, N + 1)],
    "Tenure_Months": np.random.randint(1, 61, N),
    "Monthly_Charge": np.random.uniform(20, 180, N).round(2),
    "Weekly_Usage_Hours": np.random.gamma(3, 3, N).clip(0, 35).round(1),
    "Support_Tickets": np.random.poisson(1.5, N).clip(0, 10),
    "Late_Payments": np.random.poisson(0.8, N).clip(0, 6),
    "Satisfaction_Score": np.random.randint(1, 11, N),
    "Engagement_Score": np.random.randint(1, 101, N),
    "Discount_Usage": np.random.randint(0, 6, N),
    "Payment_Failures": np.random.poisson(0.35, N).clip(0, 4),
    "Contract_Months": np.random.choice([1, 3, 12, 24], N,
                                       p=[0.45, 0.15, 0.30, 0.10])
})

# Create a synthetic churn outcome with behavioral patterns.
# This is demonstration data, not observed customer behavior.
logit = (
    -1.6
    - 0.035 * df["Tenure_Months"]
    - 0.075 * df["Weekly_Usage_Hours"]
    + 0.36 * df["Support_Tickets"]
    + 0.38 * df["Late_Payments"]
    - 0.29 * df["Satisfaction_Score"]
    - 0.018 * df["Engagement_Score"]
    + 0.65 * df["Payment_Failures"]
    + 0.65 * (df["Contract_Months"] == 1).astype(int)
    + 0.25 * (df["Monthly_Charge"] > 120).astype(int)
)

probability = 1 / (1 + np.exp(-logit))
df["Churned"] = np.random.binomial(1, probability)

FEATURES = [
    "Tenure_Months", "Monthly_Charge", "Weekly_Usage_Hours",
    "Support_Tickets", "Late_Payments", "Satisfaction_Score",
    "Engagement_Score", "Discount_Usage", "Payment_Failures",
    "Contract_Months"
]

X = df[FEATURES]
y = df["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=180,
    max_depth=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

df["Churn_Risk"] = model.predict_proba(X)[:, 1]
df["Risk_Level"] = pd.cut(
    df["Churn_Risk"],
    bins=[-0.001, 0.30, 0.60, 1.001],
    labels=["Low", "Medium", "High"]
).astype(str)

df["Estimated_Revenue_At_Risk"] = (
    df["Monthly_Charge"] * 3 * df["Churn_Risk"]
).round(2)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
accuracy = accuracy_score(y_test, model.predict(X_test))
precision = precision_score(y_test, model.predict(X_test), zero_division=0)
recall = recall_score(y_test, model.predict(X_test), zero_division=0)

# -------------------- 2. SHARED HELPERS --------------------

def money(value):
    return f"${value:,.0f}"

def make_card(label, value, note=""):
    return f"""
    <div class="metric-card">
        <div class="metric-label">{label}</div>
        <div class="metric-value">{value}</div>
        <div class="metric-note">{note}</div>
    </div>
    """

def risk_badge(score):
    if score >= 0.60:
        return "HIGH"
    elif score >= 0.30:
        return "MEDIUM"
    return "LOW"

def retention_agent(customer_id):
    row = df.loc[df["Customer_ID"] == customer_id].iloc[0]
    score = float(row["Churn_Risk"])

    factors = []
    actions = []

    if row["Support_Tickets"] >= 3:
        factors.append("Frequent support requests")
        actions.append(
            "Arrange proactive support from a named customer success "
            "representative and review unresolved issues."
        )

    if row["Satisfaction_Score"] <= 4:
        factors.append("Low satisfaction score")
        actions.append(
            "Request feedback through a short, optional conversation "
            "and address the customer's main concern."
        )

    if row["Weekly_Usage_Hours"] < 5:
        factors.append("Low product usage")
        actions.append(
            "Offer a personalized onboarding session and demonstrate "
            "features relevant to the customer's goals."
        )

    if row["Engagement_Score"] < 35:
        factors.append("Low engagement")
        actions.append(
            "Send a relevant re-engagement message with one useful "
            "feature tip rather than a generic promotional email."
        )

    if row["Late_Payments"] >= 2 or row["Payment_Failures"] >= 1:
        factors.append("Payment difficulties")
        actions.append(
            "Offer clear billing assistance and explain available "
            "payment options without applying pressure."
        )

    if row["Contract_Months"] == 1:
        factors.append("Monthly contract")
        actions.append(
            "Explain suitable plan options and their flexibility; "
            "avoid pushing a longer contract if it does not fit."
        )

    if row["Monthly_Charge"] > 120:
        factors.append("Higher monthly subscription cost")
        actions.append(
            "Review plan usage and discuss a better-fitting plan "
            "if the customer feels the current price is not justified."
        )

    if not factors:
        factors.append("No strong warning signal in the selected indicators")
        actions.append(
            "Maintain service quality, provide useful product updates, "
            "and monitor behavior for changes."
        )

    if score >= 0.60:
        priority = "URGENT — Contact promptly"
    elif score >= 0.30:
        priority = "PRIORITY — Engage proactively"
    else:
        priority = "MONITOR — Maintain engagement"

    plan = "\n".join([f"{i}. {a}" for i, a in enumerate(actions, 1)])
    factor_text = "\n".join([f"- {f}" for f in factors])

    return f"""
## Customer Retention Action Plan

**Customer:** {customer_id}

**Estimated churn risk:** {score:.1%} — **{risk_badge(score)}**

**Recommended priority:** {priority}

### Behavioral signals identified
{factor_text}

### Personalized retention actions
{plan}

### Suggested follow-up
Assign an owner, record the customer's response, and review
engagement and satisfaction over the next 2–4 weeks.

**Important:** This is an AI-assisted suggestion based on synthetic
data and model indicators. It is not a confirmed explanation of
why the customer may leave. Validate the reason with the customer.
"""

# -------------------- 3. DASHBOARD OVERVIEW --------------------

def overview():
    churn_rate = df["Churned"].mean()
    high_risk = (df["Churn_Risk"] >= 0.60).sum()
    retained = (df["Churned"] == 0).sum()
    revenue_risk = df["Estimated_Revenue_At_Risk"].sum()

    cards = f"""
    <div class="metric-grid">
      {make_card("CUSTOMERS", f"{len(df):,}", "Sample customer records")}
      {make_card("OBSERVED CHURN", f"{churn_rate:.1%}", "Synthetic historical labels")}
      {make_card("HIGH-RISK CUSTOMERS", f"{high_risk:,}", "Predicted risk ≥ 60%")}
      {make_card("REVENUE EXPOSURE", money(revenue_risk), "Illustrative 3-month weighted estimate")}
    </div>
    """

    by_risk = (
        df.groupby("Risk_Level", observed=False)
        .size().reindex(["Low", "Medium", "High"], fill_value=0)
        .rename("Customers").reset_index()
    )
    fig_risk = px.bar(
        by_risk, x="Risk_Level", y="Customers",
        color="Risk_Level",
        color_discrete_map={
            "Low": "#14B8A6", "Medium": "#A78BFA", "High": "#7C3AED"
        },
        title="Predicted Customer Risk Segments"
    )
    fig_risk.update_layout(template="plotly_white", showlegend=False)

    fig_scatter = px.scatter(
        df.sample(min(500, len(df)), random_state=42),
        x="Engagement_Score", y="Churn_Risk",
        color="Risk_Level",
        color_discrete_map={
            "Low": "#14B8A6", "Medium": "#A78BFA", "High": "#7C3AED"
        },
        hover_data=["Customer_ID", "Satisfaction_Score"],
        title="Engagement Score vs Predicted Churn Risk"
    )
    fig_scatter.update_layout(template="plotly_white")

    return cards, fig_risk, fig_scatter

# -------------------- 4. RISK ANALYZER --------------------

def analyze_customer(customer_id):
    row = df.loc[df["Customer_ID"] == customer_id].iloc[0]

    details = f"""
## Customer Risk Assessment

| Indicator | Value |
|---|---:|
| Customer ID | {row['Customer_ID']} |
| Estimated churn risk | {row['Churn_Risk']:.1%} |
| Risk category | {row['Risk_Level']} |
| Tenure | {row['Tenure_Months']} months |
| Monthly charge | {money(row['Monthly_Charge'])} |
| Weekly usage | {row['Weekly_Usage_Hours']:.1f} hours |
| Support tickets | {row['Support_Tickets']} |
| Satisfaction | {row['Satisfaction_Score']}/10 |
| Engagement | {row['Engagement_Score']}/100 |
| Payment failures | {row['Payment_Failures']} |
| Contract | {row['Contract_Months']} months |

### Interpretation
The risk score estimates the likelihood of churn according to
the trained model. It is not a guarantee that this customer will leave.
Use the behavioral indicators to guide a human review.
"""

    features = pd.DataFrame({
        "Indicator": [
            "Tenure", "Weekly usage", "Support tickets",
            "Late payments", "Satisfaction", "Engagement",
            "Payment failures"
        ],
        "Customer value": [
            row["Tenure_Months"], row["Weekly_Usage_Hours"],
            row["Support_Tickets"], row["Late_Payments"],
            row["Satisfaction_Score"], row["Engagement_Score"],
            row["Payment_Failures"]
        ]
    })

    fig = px.bar(
        features, x="Indicator", y="Customer value",
        title="Selected Customer Behavioral Indicators",
        color_discrete_sequence=["#7C3AED"]
    )
    fig.update_layout(template="plotly_white")

    return details, fig

# -------------------- 5. CUSTOMER EXPLORER --------------------

def explore_customers(risk_level, min_tenure, max_charge):
    view = df.copy()

    if risk_level != "All":
        view = view[view["Risk_Level"] == risk_level]

    view = view[
        (view["Tenure_Months"] >= min_tenure) &
        (view["Monthly_Charge"] <= max_charge)
    ]

    cols = [
        "Customer_ID", "Tenure_Months", "Monthly_Charge",
        "Weekly_Usage_Hours", "Support_Tickets",
        "Satisfaction_Score", "Engagement_Score",
        "Churn_Risk", "Risk_Level"
    ]

    result = view[cols].sort_values("Churn_Risk", ascending=False).copy()
    result["Monthly_Charge"] = result["Monthly_Charge"].round(2)
    result["Churn_Risk"] = (result["Churn_Risk"] * 100).round(1)

    return result.head(300)

# -------------------- 6. RETENTION CAMPAIGN PLANNER --------------------

def campaign_planner(segment, objective, incentive):
    subset = df.copy()

    if segment == "High-risk customers":
        subset = subset[subset["Churn_Risk"] >= 0.60]
    elif segment == "Medium-risk customers":
        subset = subset[
            (subset["Churn_Risk"] >= 0.30) &
            (subset["Churn_Risk"] < 0.60)
        ]
    elif segment == "Low-engagement customers":
        subset = subset[subset["Engagement_Score"] < 35]
    elif segment == "Low-satisfaction customers":
        subset = subset[subset["Satisfaction_Score"] <= 4]

    if objective == "Increase product usage":
        message = (
            "Subject: Get more value from your subscription\n\n"
            "Hi there,\n\n"
            "We'd love to help you get more from your subscription. "
            "Would a short guided session on features relevant to your "
            "goals be useful? Reply with a convenient time.\n\n"
            "We're here to help."
        )
        action = "Offer personalized onboarding and a relevant feature walkthrough."

    elif objective == "Improve satisfaction":
        message = (
            "Subject: How can we improve your experience?\n\n"
            "Hi there,\n\n"
            "We'd appreciate your feedback about your experience. "
            "If something isn't working as expected, our team would "
            "welcome the chance to understand and help.\n\n"
            "Thank you for being a customer."
        )
        action = "Invite feedback and resolve the customer's main issue."

    else:
        message = (
            "Subject: Let's find the right plan for you\n\n"
            "Hi there,\n\n"
            "We want your subscription to fit your needs. If your "
            "current plan no longer feels right, our team can help "
            "you review available options and explain the differences.\n\n"
            "There is no obligation to change your plan."
        )
        action = "Review plan fit, billing concerns, and suitable alternatives."

    if incentive == "No incentive":
        incentive_text = "Use service improvements and relevant support; no discount."
    elif incentive == "Optional discount":
        incentive_text = (
            "Consider a modest, approved discount only after reviewing "
            "eligibility, margin, and whether price is a genuine concern."
        )
    else:
        incentive_text = (
            "Consider a non-monetary benefit, such as guided onboarding "
            "or a relevant service consultation."
        )

    return f"""
## Retention Campaign Plan

**Target segment:** {segment}

**Customers matching the segment:** {len(subset):,}

**Primary objective:** {objective}

### Recommended action
{action}

### Incentive guidance
{incentive_text}

### Draft customer message
{message}

### Measurement plan
- Track customer replies and engagement.
- Monitor usage and satisfaction before and after outreach.
- Measure actual churn among contacted and comparable eligible customers.
- Track retention costs and margin impact.

Do not assume this campaign will prevent churn. Test the approach
and compare outcomes before scaling it.
"""

# -------------------- 7. STYLING --------------------

css = """
.gradio-container {
    background: #F7F5FF !important;
    font-family: Inter, Arial, sans-serif;
}
#main-header {
    background: linear-gradient(120deg, #4C1D95, #7C3AED, #0F766E);
    color: white;
    padding: 28px;
    border-radius: 18px;
    margin-bottom: 16px;
}
#main-header h1 { color: white !important; margin-bottom: 6px; }
#main-header p { color: #EDE9FE !important; }
.metric-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(140px, 1fr));
    gap: 12px;
    margin: 12px 0;
}
.metric-card {
    background: white;
    border: 1px solid #DDD6FE;
    border-top: 4px solid #7C3AED;
    border-radius: 14px;
    padding: 18px;
    min-height: 100px;
}
.metric-label {
    color: #6D28D9;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.5px;
}
.metric-value {
    color: #2E1065;
    font-size: 27px;
    font-weight: 750;
    margin: 8px 0;
}
.metric-note { color: #64748B; font-size: 11px; }
"""

# -------------------- 8. BUILD INTERACTIVE APP --------------------

customer_ids = df["Customer_ID"].tolist()

with gr.Blocks(title="Customer Churn & Retention Intelligence") as demo:

    gr.HTML("""
    <div id="main-header">
        <h1>Customer Churn & Retention Intelligence</h1>
        <p>AI-assisted churn risk analysis, customer segmentation,
        and personalized retention planning</p>
    </div>
    """)

    gr.Markdown(
        "Explore churn signals, assess individual customers, "
        "and create targeted retention plans."
    )

    with gr.Tabs():

        with gr.Tab("Executive Overview"):
            overview_cards = gr.HTML()
            with gr.Row():
                risk_chart = gr.Plot(label="Risk Segments")
                engagement_chart = gr.Plot(label="Engagement & Risk")

            refresh = gr.Button("Refresh Overview", variant="primary")
            refresh.click(
                overview,
                outputs=[overview_cards, risk_chart, engagement_chart]
            )
            demo.load(
                overview,
                outputs=[overview_cards, risk_chart, engagement_chart]
            )

        with gr.Tab("Customer Risk Analyzer"):
            gr.Markdown("### Assess an individual customer")
            customer_input = gr.Dropdown(
                choices=customer_ids,
                value=customer_ids[0],
                label="Select Customer ID",
                interactive=True
            )
            analyze_button = gr.Button("Analyze Customer", variant="primary")
            with gr.Row():
                risk_details = gr.Markdown()
                customer_chart = gr.Plot()

            analyze_button.click(
                analyze_customer,
                inputs=customer_input,
                outputs=[risk_details, customer_chart]
            )
            customer_input.change(
                analyze_customer,
                inputs=customer_input,
                outputs=[risk_details, customer_chart]
            )

        with gr.Tab("Retention Strategy Agent"):
            gr.Markdown(
                "### Generate a personalized retention plan\n"
                "Choose a customer to identify relevant warning signals "
                "and recommended actions."
            )
            agent_customer = gr.Dropdown(
                choices=customer_ids,
                value=customer_ids[0],
                label="Customer ID"
            )
            agent_button = gr.Button(
                "Generate Retention Plan", variant="primary"
            )
            agent_output = gr.Markdown()

            agent_button.click(
                retention_agent,
                inputs=agent_customer,
                outputs=agent_output
            )
            agent_customer.change(
                retention_agent,
                inputs=agent_customer,
                outputs=agent_output
            )

        with gr.Tab("Retention Campaign Planner"):
            gr.Markdown(
                "### Build a targeted outreach plan\n"
                "Create a sample message and recommended action for a segment."
            )
            segment_input = gr.Dropdown(
                choices=[
                    "High-risk customers",
                    "Medium-risk customers",
                    "Low-engagement customers",
                    "Low-satisfaction customers",
                    "All customers"
                ],
                value="High-risk customers",
                label="Target Segment"
            )
            objective_input = gr.Radio(
                choices=[
                    "Increase product usage",
                    "Improve satisfaction",
                    "Review subscription plan"
                ],
                value="Improve satisfaction",
                label="Campaign Objective"
            )
            incentive_input = gr.Radio(
                choices=[
                    "No incentive",
                    "Optional discount",
                    "Non-monetary benefit"
                ],
                value="No incentive",
                label="Incentive Approach"
            )
            campaign_button = gr.Button(
                "Create Campaign Plan", variant="primary"
            )
            campaign_output = gr.Markdown()

            campaign_button.click(
                campaign_planner,
                inputs=[
                    segment_input, objective_input, incentive_input
                ],
                outputs=campaign_output
            )

        with gr.Tab("Customer Explorer"):
            gr.Markdown(
                "### Filter and explore customers\n"
                "Risk scores are model estimates, not confirmed outcomes."
            )
            with gr.Row():
                filter_risk = gr.Dropdown(
                    choices=["All", "Low", "Medium", "High"],
                    value="All",
                    label="Risk Segment"
                )
                filter_tenure = gr.Slider(
                    minimum=1, maximum=60, value=1, step=1,
                    label="Minimum Tenure (months)"
                )
                filter_charge = gr.Slider(
                    minimum=20, maximum=180, value=180, step=5,
                    label="Maximum Monthly Charge ($)"
                )

            explore_button = gr.Button(
                "Apply Filters", variant="primary"
            )
            customer_table = gr.Dataframe(
                label="Filtered Customers (risk percentage)",
                interactive=False
            )
            explore_button.click(
                explore_customers,
                inputs=[filter_risk, filter_tenure, filter_charge],
                outputs=customer_table
            )
            for component in [filter_risk, filter_tenure, filter_charge]:
                component.change(
                    explore_customers,
                    inputs=[filter_risk, filter_tenure, filter_charge],
                    outputs=customer_table
                )
            demo.load(
                explore_customers,
                inputs=[filter_risk, filter_tenure, filter_charge],
                outputs=customer_table
            )

        with gr.Tab("Model & Methodology"):
            gr.Markdown(f"""
## Model information

**Algorithm:** Random Forest Classifier

**Training records:** {len(X_train):,}

**Test records:** {len(X_test):,}

| Evaluation Metric | Test Result |
|---|---:|
| Accuracy | {accuracy:.1%} |
| ROC-AUC | {auc:.3f} |
| Precision | {precision:.1%} |
| Recall | {recall:.1%} |

### Features used
Tenure, monthly charge, weekly usage, support tickets, late payments,
satisfaction, engagement, discount usage, payment failures, and contract length.

### Important limitations
- Customer records and churn labels are synthetically generated.
- Metrics describe performance on a held-out synthetic test set only.
- Feature relationships were simulated and may not reflect real customers.
- The risk score is an estimate, not proof of a customer's intention to leave.
- Revenue exposure is a simplified illustrative estimate, not a financial forecast.
- Retention actions are suggestions; validate them before contacting customers.
- Real deployment requires real data, validation, privacy safeguards,
  and monitoring for model drift.
""")

    gr.Markdown(
        "Customer Churn & Retention Intelligence | "
        "AI-assisted prototype | Synthetic data"
    )

# Gradio 6: theme and CSS are passed to launch().
demo.launch(
    share=False,
    debug=False,
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="teal",
        neutral_hue="slate"
    ),
    css=css
)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [2]:

%pip install --quiet "numpy==2.2.6" "pandas==2.2.3" "scipy>=1.15,<1.16" "scikit-learn>=1.6" "plotly>=6" "gradio>=5,<7"


In [3]:

import numpy as np
import pandas as pd
import scipy
import sklearn
import plotly.express as px
import gradio as gr

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Plotly and Gradio imported successfully!")


AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [4]:
%pip install --force-reinstall --no-cache-dir \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "pandas==2.2.3" \
    "scikit-learn==1.7.2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 213.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 151.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 147.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 148.7 MB/s  0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.5
    Uninstalling tzdata-2026.5:
      Successfully uninstalled tzdata-2026.5
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.7.0
    Uninstalling threadpoolctl-3.7.0:
      Successfully uninstalled threadpoolctl-3.7.0
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 

In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import plotly.express as px
import gradio as gr

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("All imports successful!")

NumPy: 2.2.6
Pandas: 2.2.3
SciPy: 1.15.3
Scikit-learn: 1.7.2
All imports successful!


In [2]:
%pip install -q "gradio>=5,<7" plotly

In [5]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# =========================================================
# 1. PROJECT CONFIGURATION
# =========================================================

np.random.seed(42)

APP_TITLE = "Customer Churn & Retention Intelligence"
PURPLE = "#7657D7"
TEAL = "#16B8A6"

# =========================================================
# 2. GENERATE SYNTHETIC CUSTOMER DATA
# =========================================================
# Demonstration data only; not real customer information.

n = 1800

customers = pd.DataFrame({
    "Customer_ID": [f"CUST-{i:04d}" for i in range(1, n + 1)],
    "Tenure_Months": np.random.randint(1, 73, n),
    "Monthly_Charges": np.round(np.random.uniform(20, 220, n), 2),
    "Usage_Hours": np.round(np.random.uniform(1, 100, n), 1),
    "Support_Tickets": np.random.poisson(2.0, n).clip(0, 12),
    "Payment_Delays": np.random.poisson(1.2, n).clip(0, 8),
    "Satisfaction_Score": np.random.randint(1, 11, n),
    "Contract_Type": np.random.choice(
        ["Monthly", "Quarterly", "Annual"],
        n,
        p=[0.55, 0.20, 0.25]
    ),
    "Payment_Method": np.random.choice(
        ["Credit Card", "Bank Transfer", "Digital Wallet"],
        n
    )
})

# Create illustrative churn labels from customer behavior.
contract_penalty = customers["Contract_Type"].map({
    "Monthly": 1.0,
    "Quarterly": 0.4,
    "Annual": -0.8
})

log_odds = (
    -0.8
    - 0.025 * customers["Tenure_Months"]
    + 0.22 * customers["Support_Tickets"]
    + 0.30 * customers["Payment_Delays"]
    - 0.25 * customers["Satisfaction_Score"]
    - 0.012 * customers["Usage_Hours"]
    + 0.65 * contract_penalty
    + 0.004 * (customers["Monthly_Charges"] - 100)
)

probability = 1 / (1 + np.exp(-log_odds))
probability = np.clip(probability, 0.02, 0.95)

customers["Churned"] = np.random.binomial(1, probability)

# =========================================================
# 3. PREPARE AND TRAIN THE MODEL
# =========================================================

feature_cols = [
    "Tenure_Months",
    "Monthly_Charges",
    "Usage_Hours",
    "Support_Tickets",
    "Payment_Delays",
    "Satisfaction_Score",
    "Contract_Type"
]

def prepare_features(data):
    result = data[feature_cols].copy()
    result["Contract_Type"] = result["Contract_Type"].map({
        "Monthly": 0,
        "Quarterly": 1,
        "Annual": 2
    })
    return result.astype(float)

X = prepare_features(customers)
y = customers["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, test_predictions)
auc = roc_auc_score(y_test, test_probabilities)

customers["Churn_Risk"] = model.predict_proba(X)[:, 1] * 100

def risk_label(value):
    if value >= 65:
        return "High Risk"
    elif value >= 35:
        return "Medium Risk"
    return "Low Risk"

customers["Risk_Segment"] = customers["Churn_Risk"].apply(risk_label)

def money(value):
    return f"${value:,.2f}"

# =========================================================
# 4. DASHBOARD CHARTS
# =========================================================

risk_counts = (
    customers["Risk_Segment"]
    .value_counts()
    .reindex(["High Risk", "Medium Risk", "Low Risk"])
    .fillna(0)
    .astype(int)
    .rename_axis("Risk Segment")
    .reset_index(name="Customers")
)

fig_risk = px.pie(
    risk_counts,
    names="Risk Segment",
    values="Customers",
    hole=0.55,
    title="Customer Risk Distribution",
    color="Risk Segment",
    color_discrete_map={
        "High Risk": "#E76F78",
        "Medium Risk": "#F2B84B",
        "Low Risk": "#16B8A6"
    }
)
fig_risk.update_layout(
    template="plotly_white",
    legend_title_text="",
    margin=dict(t=60, b=20, l=20, r=20)
)

fig_contract = px.bar(
    customers.groupby("Contract_Type", as_index=False)
    .agg(Churn_Rate=("Churned", "mean"), Customers=("Customer_ID", "count")),
    x="Contract_Type",
    y="Churn_Rate",
    color="Contract_Type",
    title="Observed Churn Rate by Contract",
    labels={"Churn_Rate": "Churn Rate", "Contract_Type": "Contract"},
    text_auto=".0%"
)
fig_contract.update_layout(
    template="plotly_white",
    showlegend=False,
    yaxis_tickformat=".0%",
    margin=dict(t=60, b=20, l=20, r=20)
)

fig_tenure = px.histogram(
    customers,
    x="Tenure_Months",
    color="Risk_Segment",
    nbins=24,
    title="Customer Tenure and Risk",
    color_discrete_map={
        "High Risk": "#E76F78",
        "Medium Risk": "#F2B84B",
        "Low Risk": "#16B8A6"
    },
    labels={"Tenure_Months": "Tenure (months)"}
)
fig_tenure.update_layout(
    template="plotly_white",
    margin=dict(t=60, b=20, l=20, r=20)
)

# =========================================================
# 5. RETENTION RECOMMENDATION LOGIC
# =========================================================

def build_recommendation(risk, tenure, satisfaction, tickets, delays, contract):
    actions = []

    if satisfaction <= 4:
        actions.append(
            "Arrange a customer-success check-in to understand dissatisfaction."
        )

    if tickets >= 4:
        actions.append(
            "Review unresolved support cases and provide a clear resolution plan."
        )

    if delays >= 3:
        actions.append(
            "Offer payment reminders and help resolve billing or payment difficulties."
        )

    if tenure <= 6:
        actions.append(
            "Provide onboarding guidance and demonstrate useful product features."
        )

    if contract == "Monthly":
        actions.append(
            "Explain the benefits of longer-term plans without pressuring the customer."
        )

    if not actions:
        actions.append(
            "Maintain engagement with relevant product tips and periodic satisfaction checks."
        )

    if risk >= 65:
        priority = "HIGH — contact promptly"
    elif risk >= 35:
        priority = "MEDIUM — follow up proactively"
    else:
        priority = "LOW — maintain engagement"

    return (
        f"### Estimated churn risk: {risk:.1f}%\n"
        f"**Priority:** {priority}\n\n"
        "**Suggested retention actions**\n\n"
        + "\n".join(f"- {action}" for action in actions)
        + "\n\n*These are model-generated suggestions for a demonstration, "
          "not guarantees of customer behavior.*"
    )

# =========================================================
# 6. INTERACTIVE FUNCTIONS
# =========================================================

def analyze_customer(tenure, charges, usage, tickets, delays, satisfaction, contract):
    record = pd.DataFrame([{
        "Tenure_Months": tenure,
        "Monthly_Charges": charges,
        "Usage_Hours": usage,
        "Support_Tickets": tickets,
        "Payment_Delays": delays,
        "Satisfaction_Score": satisfaction,
        "Contract_Type": contract
    }])

    risk = float(model.predict_proba(prepare_features(record))[0, 1] * 100)

    recommendation = build_recommendation(
        risk, tenure, satisfaction, tickets, delays, contract
    )

    return (
        f"{risk:.1f}%",
        risk_label(risk),
        recommendation
    )

def generate_campaign(segment, channel, offer):
    campaign_messages = {
        "High Risk": {
            "Email": (
                "Subject: Let's make your experience better\n\n"
                "We'd love to understand how we can improve your experience. "
                "Reply to this email or schedule a conversation with our team. "
                "We'll work with you to address any concerns."
            ),
            "SMS": (
                "We value your feedback. If something isn't working for you, "
                "our support team is here to help. Reply to arrange a follow-up."
            ),
            "In-App Message": (
                "Need a better experience? Tell us what's missing and explore "
                "the support options available to you."
            )
        },
        "Medium Risk": {
            "Email": (
                "Subject: Get more value from your subscription\n\n"
                "Explore features and tips tailored to help you get more from "
                "your subscription. Our team is available if you need help."
            ),
            "SMS": (
                "Discover more ways to get value from your subscription. "
                "Contact our team for tips or assistance."
            ),
            "In-App Message": (
                "Make the most of your subscription: explore helpful features "
                "and discover tips selected for your needs."
            )
        },
        "Low Risk": {
            "Email": (
                "Subject: What's new for you\n\n"
                "Thanks for being with us. Take a look at our latest features, "
                "helpful tips, and updates designed to improve your experience."
            ),
            "SMS": (
                "Thanks for being a customer! Explore our latest features and "
                "tips to get even more from your subscription."
            ),
            "In-App Message": (
                "See what's new! Explore features, updates, and tips to make "
                "your experience even better."
            )
        }
    }

    message = campaign_messages[segment][channel]

    offer_text = {
        "No offer": "No discount or incentive is included.",
        "Personalized support": "Offer a personalized support session where appropriate.",
        "10% discount": "Consider a 10% discount, subject to eligibility and business approval."
    }[offer]

    return (
        f"## {segment} Retention Campaign\n"
        f"**Channel:** {channel}\n\n"
        f"{message}\n\n"
        f"**Incentive / follow-up:** {offer_text}\n\n"
        "**Measurement:** Track delivery, engagement, responses, retention, "
        "and unsubscribe rates. Compare results with an appropriate control group."
    )

def explore_customers(segment, min_tenure, max_charges):
    data = customers[
        (customers["Risk_Segment"] == segment)
        & (customers["Tenure_Months"] >= min_tenure)
        & (customers["Monthly_Charges"] <= max_charges)
    ].copy()

    columns = [
        "Customer_ID", "Tenure_Months", "Monthly_Charges",
        "Satisfaction_Score", "Support_Tickets", "Payment_Delays",
        "Contract_Type", "Churn_Risk", "Risk_Segment"
    ]

    data = data[columns].sort_values("Churn_Risk", ascending=False).head(100)
    data["Churn_Risk"] = data["Churn_Risk"].round(1)

    return data, f"Showing {len(data)} matching customers (maximum 100)."

def show_feature_importance():
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=True)

    fig = px.bar(
        importance,
        x="Importance",
        y="Feature",
        orientation="h",
        title="Model Feature Importance",
        color="Importance",
        color_continuous_scale=["#B7A6F6", "#7657D7"]
    )
    fig.update_layout(template="plotly_white", margin=dict(t=60, b=20))
    return fig



# =========================================================
# 7. DASHBOARD DESIGN
# =========================================================

css = """
.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
}
#main-header {
    padding: 26px;
    border-radius: 18px;
    background: linear-gradient(120deg, #7657D7, #16B8A6);
    color: white;
    margin-bottom: 18px;
}
#main-header h1 {
    color: white !important;
    font-size: 30px !important;
}
#main-header p {
    color: white !important;
    font-size: 15px;
}
.metric-card {
    border-radius: 14px;
    padding: 16px;
}
"""

theme = gr.themes.Soft(
    primary_hue="purple",
    secondary_hue="teal",
    neutral_hue="slate"
)


# =========================================================
# 8. BUILD THE APP
# =========================================================

with gr.Blocks(theme=theme, css=css, title=APP_TITLE) as demo:

    gr.HTML("""
    <div id="main-header">
      <h1>Customer Churn & Retention Intelligence</h1>
      <p>Understand customer risk, explore behavioral signals, and plan
      more targeted retention actions.</p>
    </div>
    """)

    gr.Markdown(
        "**Demonstration notice:** This dashboard uses synthetic data and a "
        "machine-learning model trained on illustrative churn labels. "
        "It is not connected to a real company's customer database."
    )

    with gr.Tabs():

        # -------------------------------------------------
        # TAB 1: EXECUTIVE OVERVIEW
        # -------------------------------------------------
        with gr.Tab("Executive Overview"):
            gr.Markdown("## Business Snapshot")

            with gr.Row():
                gr.Number(
                    value=len(customers),
                    label="Customers in Dataset",
                    interactive=False
                )
                gr.Number(
                    value=round(customers["Churned"].mean() * 100, 1),
                    label="Synthetic Churn Rate (%)",
                    interactive=False
                )
                gr.Number(
                    value=int((customers["Risk_Segment"] == "High Risk").sum()),
                    label="Customers Flagged High Risk",
                    interactive=False
                )
                gr.Number(
                    value=round(accuracy * 100, 1),
                    label="Test Accuracy (%)",
                    interactive=False
                )

            with gr.Row():
                gr.Plot(value=fig_risk, label="Risk Distribution")
                gr.Plot(value=fig_contract, label="Contract Analysis")

            gr.Plot(value=fig_tenure, label="Tenure Analysis")

            gr.Markdown(
                "### How to read this dashboard\n"
                "- **High risk:** prioritize a timely, helpful follow-up.\n"
                "- **Medium risk:** investigate potential friction and engage proactively.\n"
                "- **Low risk:** maintain satisfaction and product engagement.\n\n"
                "Risk thresholds are illustrative and should be calibrated before real use."
            )

        # -------------------------------------------------
        # TAB 2: CUSTOMER RISK ANALYZER
        # -------------------------------------------------
        with gr.Tab("Customer Risk Analyzer"):
            gr.Markdown(
                "## Analyze an Individual Customer\n"
                "Adjust the inputs to estimate a customer's churn risk."
            )

            with gr.Row():
                with gr.Column():
                    tenure = gr.Slider(
                        1, 72, value=12, step=1,
                        label="Customer Tenure (months)"
                    )
                    charges = gr.Slider(
                        20, 220, value=90, step=5,
                        label="Monthly Charges ($)"
                    )
                    usage = gr.Slider(
                        1, 100, value=35, step=1,
                        label="Monthly Usage (hours)"
                    )
                    tickets = gr.Slider(
                        0, 12, value=3, step=1,
                        label="Support Tickets"
                    )
                    delays = gr.Slider(
                        0, 8, value=1, step=1,
                        label="Payment Delays"
                    )
                    satisfaction = gr.Slider(
                        1, 10, value=6, step=1,
                        label="Satisfaction Score (1–10)"
                    )
                    contract = gr.Dropdown(
                        ["Monthly", "Quarterly", "Annual"],
                        value="Monthly",
                        label="Contract Type"
                    )
                    analyze_btn = gr.Button(
                        "Analyze Churn Risk",
                        variant="primary"
                    )

                with gr.Column():
                    risk_output = gr.Textbox(
                        label="Estimated Churn Probability",
                        interactive=False
                    )
                    segment_output = gr.Textbox(
                        label="Risk Segment",
                        interactive=False
                    )
                    recommendation_output = gr.Markdown(
                        label="Suggested Retention Actions"
                    )

            analyze_btn.click(
                fn=analyze_customer,
                inputs=[
                    tenure, charges, usage, tickets,
                    delays, satisfaction, contract
                ],
                outputs=[
                    risk_output, segment_output, recommendation_output
                ]
            )

        # -------------------------------------------------
        # TAB 3: RETENTION STRATEGY AGENT
        # -------------------------------------------------
        with gr.Tab("Retention Strategy Agent"):
            gr.Markdown(
                "## Retention Strategy Assistant\n"
                "Generate practical next steps based on a customer's "
                "risk score and behavioral warning signals."
            )

            with gr.Row():
                strategy_risk = gr.Slider(
                    0, 100, value=75, step=1,
                    label="Estimated Churn Risk (%)"
                )
                strategy_tenure = gr.Slider(
                    1, 72, value=4, step=1,
                    label="Tenure (months)"
                )

            with gr.Row():
                strategy_satisfaction = gr.Slider(
                    1, 10, value=4, step=1,
                    label="Satisfaction (1–10)"
                )
                strategy_tickets = gr.Slider(
                    0, 12, value=5, step=1,
                    label="Support Tickets"
                )

            with gr.Row():
                strategy_delays = gr.Slider(
                    0, 8, value=3, step=1,
                    label="Payment Delays"
                )
                strategy_contract = gr.Dropdown(
                    ["Monthly", "Quarterly", "Annual"],
                    value="Monthly",
                    label="Contract Type"
                )

            strategy_btn = gr.Button(
                "Generate Retention Strategy",
                variant="primary"
            )
            strategy_output = gr.Markdown()

            strategy_btn.click(
                fn=build_recommendation,
                inputs=[
                    strategy_risk, strategy_tenure,
                    strategy_satisfaction, strategy_tickets,
                    strategy_delays, strategy_contract
                ],
                outputs=strategy_output
            )

        # -------------------------------------------------
        # TAB 4: RETENTION CAMPAIGN PLANNER
        # -------------------------------------------------
        with gr.Tab("Retention Campaign Planner"):
            gr.Markdown(
                "## Retention Campaign Planner\n"
                "Create a draft outreach message for a selected customer segment."
            )

            with gr.Row():
                campaign_segment = gr.Dropdown(
                    ["High Risk", "Medium Risk", "Low Risk"],
                    value="High Risk",
                    label="Customer Segment"
                )
                campaign_channel = gr.Dropdown(
                    ["Email", "SMS", "In-App Message"],
                    value="Email",
                    label="Communication Channel"
                )
                campaign_offer = gr.Dropdown(
                    ["No offer", "Personalized support", "10% discount"],
                    value="Personalized support",
                    label="Offer / Follow-up"
                )

            campaign_btn = gr.Button(
                "Generate Campaign Draft",
                variant="primary"
            )
            campaign_output = gr.Markdown()

            campaign_btn.click(
                fn=generate_campaign,
                inputs=[
                    campaign_segment, campaign_channel, campaign_offer
                ],
                outputs=campaign_output
            )

        # -------------------------------------------------
        # TAB 5: CUSTOMER EXPLORER
        # -------------------------------------------------
        with gr.Tab("Customer Explorer"):
            gr.Markdown(
                "## Explore Customer Segments\n"
                "Filter the synthetic customer dataset to identify records "
                "for further review."
            )

            with gr.Row():
                explorer_segment = gr.Dropdown(
                    ["High Risk", "Medium Risk", "Low Risk"],
                    value="High Risk",
                    label="Risk Segment"
                )
                explorer_tenure = gr.Slider(
                    1, 72, value=1, step=1,
                    label="Minimum Tenure (months)"
                )
                explorer_charges = gr.Slider(
                    20, 220, value=220, step=5,
                    label="Maximum Monthly Charges ($)"
                )

            explorer_btn = gr.Button(
                "Explore Customers",
                variant="primary"
            )
            explorer_status = gr.Markdown()
            explorer_table = gr.Dataframe(
                label="Matching Customer Records",
                interactive=False
            )

            explorer_btn.click(
                fn=explore_customers,
                inputs=[
                    explorer_segment, explorer_tenure, explorer_charges
                ],
                outputs=[explorer_table, explorer_status]
            )

            demo_note = gr.Markdown(
                "Customer IDs and records are synthetic. Do not treat these "
                "records as real customers."
            )

        # -------------------------------------------------
        # TAB 6: MODEL & METHODOLOGY
        # -------------------------------------------------
        with gr.Tab("Model & Methodology"):
            gr.Markdown("## Model Overview")

            with gr.Row():
                gr.Number(
                    value=round(accuracy * 100, 2),
                    label="Test Accuracy (%)",
                    interactive=False
                )
                gr.Number(
                    value=round(auc, 3),
                    label="Test ROC-AUC",
                    interactive=False
                )

            gr.Markdown(
                "### Workflow\n"
                "1. Generate a synthetic customer dataset.\n"
                "2. Prepare tenure, charges, usage, support, payment, satisfaction, and contract features.\n"
                "3. Split the data into training and testing sets.\n"
                "4. Train a Random Forest classifier.\n"
                "5. Estimate churn probabilities for customers.\n"
                "6. Group customers into low-, medium-, and high-risk segments.\n"
                "7. Generate suggested retention actions and campaign drafts.\n\n"
                "### Important limitations\n"
                "- Data and churn labels are simulated, not sourced from a real business.\n"
                "- Model metrics describe performance on this synthetic dataset only.\n"
                "- Feature importance and risk estimates do not establish causation.\n"
                "- Validate the model on real, representative historical data before business use."
            )

            gr.Plot(value=show_feature_importance())

# =========================================================
# 9. LAUNCH DASHBOARD
# =========================================================

print("Model training complete.")
print(f"Test accuracy: {accuracy:.1%}")
print(f"Test ROC-AUC: {auc:.3f}")
print("Launching dashboard...")

demo.launch(share=True, debug=False)


Model training complete.
Test accuracy: 85.6%
Test ROC-AUC: 0.762
Launching dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a00dbd7e97d61e21c7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
